In [2]:
import json
import time
import os
import re
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed
from openai import OpenAI
from tqdm.auto import tqdm
from dotenv import load_dotenv

load_dotenv()
# 1. Fetch key from environment variables
api_key_env = os.environ.get("OPENROUTER_API_KEY")


client = OpenAI(
  base_url="https://openrouter.ai/api/v1",
  api_key=api_key_env,
  timeout=180.0 # Increased timeout for the larger V3.1 model
)


# Model: Qwen 3.5 9B Instruct
MODEL_ID = "deepseek/deepseek-chat-v3.1"

In [3]:
def get_model_response(prompt):
    """
    High-fidelity inference for DeepSeek V3.1.
    We allow up to 4,096 tokens to ensure the research answers are never truncated.
    """
    system_prompt = (
        "You are a helpful assistant. Follow instructions strictly. "
        "Provide your answer in plain text ONLY unless stated otherwise. "
    )

    for attempt in range(3):
        try:
            completion = client.chat.completions.create(
                model=MODEL_ID,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": prompt},
                ],
                temperature=0.0, # Mandatory for research consistency
                max_tokens=4096,
                extra_headers={
                    "HTTP-Referer": "http://localhost",
                    "X-Title": "ANOVA Prompt Syntax Study",
                }
            )
            raw_content = completion.choices[0].message.content.strip()

            # Post-processing: DeepSeek V3.1 is very good, but we keep the cleaner
            # just in case it triggers a reasoning block.
            cleaned = re.sub(r"<think>.*?</think>", "", raw_content, flags=re.DOTALL)
            cleaned = re.sub(r"(?i)(Thinking Process|Thought|Analysis|Reasoning|Action):.*?\n\n", "", cleaned, flags=re.DOTALL)

            return cleaned.strip()

        except Exception as e:
            time.sleep(5 + (2 ** attempt)) # Backoff for API stability

    return "API_ERROR: Max retries exceeded"

In [4]:
file_lock = threading.Lock()

def process_variant(key, fmt, prompt_text):
    """ Worker function for individual variations """
    return key, fmt, get_model_response(prompt_text)

def run_anova_study(input_path, output_path, max_workers=15):
    """
    Parallelized execution using DeepSeek V3.1.
    max_workers is set to 15 to stay within stable RPM (Requests Per Minute) limits.
    """
    if not os.path.exists(input_path):
        print(f"Error: {input_path} not found.")
        return

    with open(input_path, 'r', encoding='utf-8') as f:
        master_data = json.load(f)

    queue = [e for e in master_data if str(e.get('status','')).lower() == 'done']
    results_map = {}

    # RESUME: Skips everything already completed
    if os.path.exists(output_path):
        try:
            with open(output_path, 'r', encoding='utf-8') as f:
                existing = json.load(f)
                for item in existing:
                    results_map[str(item['key'])] = item
                print(f"✅ Resuming: {len(results_map)} keys already finished.")
        except:
            print("Starting fresh results file.")

    # Flatten the queue into individual tasks
    tasks = []
    for entry in queue:
        k = str(entry['key'])
        if k in results_map: continue
        for fmt, text in entry['variations'].items():
            if text:
                tasks.append((k, fmt, text, entry))

    if not tasks:
        print("All prompts processed.")
        return

    print(f"🚀 Launching DeepSeek V3.1 Run | {len(tasks)} variations | {max_workers} Workers")
    temp_results = {}

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = [executor.submit(process_variant, t[0], t[1], t[2]) for t in tasks]

        for future in tqdm(as_completed(futures), total=len(tasks), desc="Generations"):
            key, fmt, response = future.result()

            with file_lock:
                if key not in temp_results:
                    orig = next(e for e in queue if str(e['key']) == key)
                    temp_results[key] = {
                        "key": orig['key'],
                        "instruction_id_list": orig['instruction_id_list'],
                        "kwargs": orig['kwargs'],
                        "responses": {}
                    }

                temp_results[key]["responses"][fmt] = response

                # Check if this key has all variations finished
                orig_vars = next(e['variations'] for e in queue if str(e['key']) == key)
                expected_count = len([v for v in orig_vars.values() if v])

                if len(temp_results[key]["responses"]) == expected_count:
                    results_map[key] = temp_results.pop(key)
                    with open(output_path, 'w', encoding='utf-8') as f:
                        json.dump(list(results_map.values()), f, indent=2)

In [5]:
test_prompt = "Explain why XML tags help structured prompts in exactly one sentence."
print("Testing API Response...")
start = time.time()
response = get_model_response(test_prompt)
print(f"Response: {response}")
print(f"Time: {time.time() - start:.2f}s")

Testing API Response...
Response: XML tags help structured prompts by clearly delineating different components, ensuring precise interpretation and consistent formatting for the AI.
Time: 2.99s


In [6]:
# Ensure these files are in your PyCharm project folder
INPUT_JSON = 'data/IfEvalTestFile.json'
OUTPUT_JSON = 'results/IfEvalTestResults.json'



# max_workers=10 means 10 entries (50 variations) can be processed at the same time.
# Start with 10; increase to 20 if your OpenRouter rate limit allows it.
run_anova_study(INPUT_JSON, OUTPUT_JSON, max_workers=15)

🚀 Launching DeepSeek V3.1 Run | 2705 variations | 15 Workers


Generations:   0%|          | 0/2705 [00:00<?, ?it/s]